
## 1. Выбор начальных условий

### 1a. Выбор набора данных и обоснование
Выбран набор данных Building Dataset для задачи многоклассовой классификации изображений бетона по типам дефектов.

### 1b. Выбор метрик качества и обоснование
Для оценки качества используются Accuracy и F1 macro.
Accuracy показывает долю верных ответов.
F1 macro учитывает каждый класс одинаково и позволяет корректнее оценивать качество при возможном дисбалансе классов.
Дополнительно выводятся отчёт по классам и матрица ошибок.

## 2. Создание бейзлайна и оценка качества

Подготовка окружения: импорт библиотек, фиксация seed, подключение Google Drive в Colab.

In [1]:
import random
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)

random_state = 42
random.seed(random_state)
np.random.seed(random_state)
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', repr(e))

torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
device: cuda
Mounted at /content/drive


### 2a. Обучение моделей из torchvision

Подготовка данных: путь к датасету, сбор размеченных изображений, разбиение на train и val.
Далее обучаются две модели из torchvision: сверточная ResNet18 и трансформер ViT B 16. Если ViT недоступна, используется Swin T.

In [2]:
dataset_dir = Path('/content/drive/MyDrive/DATASETS/Building_Dataset')
print('Dataset dir:', dataset_dir)
assert dataset_dir.exists(), 'Папка датасета не найдена. Проверь путь dataset_dir.'

allowed_ext = {'.jpg', '.jpeg', '.png', '.bmp'}

if (dataset_dir / 'Wall').exists():
    print('Folder Wall found. It will be ignored.')

class_dirs = sorted([p for p in dataset_dir.iterdir() if p.is_dir()])
class_names = [p.name for p in class_dirs]
num_classes = len(class_names)

print('Classes:', class_names)
assert num_classes >= 2, 'Не получилось найти папки классов. Проверь структуру датасета.'

samples = []
for class_idx, class_name in enumerate(class_names):
    img_dir = dataset_dir / class_name
    img_paths = [p for p in img_dir.rglob('*') if p.is_file() and p.suffix.lower() in allowed_ext]
    for p in img_paths:
        samples.append((str(p), class_idx))
    print(f'{class_name}: {len(img_paths)} images')

all_paths = [p for p, _ in samples]
all_labels = [y for _, y in samples]
print('Total labeled images:', len(all_paths))
assert len(all_paths) > 0, 'Не нашлись изображения. Проверь путь dataset_dir и расширения файлов.'

try:
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        all_paths,
        all_labels,
        test_size=0.2,
        random_state=random_state,
        stratify=all_labels,
    )
except ValueError as e:
    print('Stratified split failed, fallback to non-stratified:', repr(e))
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        all_paths,
        all_labels,
        test_size=0.2,
        random_state=random_state,
        stratify=None,
    )

print('Train size:', len(train_paths))
print('Val size:', len(val_paths))

Dataset dir: /content/drive/MyDrive/DATASETS/Building_Dataset
Classes: ['algae', 'major_crack', 'minor_crack', 'normal', 'peeling', 'spalling', 'stain']
algae: 520 images
major_crack: 480 images
minor_crack: 380 images
normal: 450 images
peeling: 420 images
spalling: 350 images
stain: 380 images
Total labeled images: 2980
Train size: 2384
Val size: 596


#### Подготовка датасета и DataLoader

Используется Dataset по спискам путей и меток.
Нормализация как для ImageNet, так как используются предобученные веса.

In [3]:
class ImagePathsDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        y = int(self.labels[idx])
        return img, y

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

image_size = 224

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(image_size, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

val_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

def make_loaders(batch_size, num_workers=2):
    train_ds = ImagePathsDataset(train_paths, train_labels, transform=train_tfms)
    val_ds = ImagePathsDataset(val_paths, val_labels, transform=val_tfms)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader

train_loader_tmp, val_loader_tmp = make_loaders(batch_size=8)
x_tmp, y_tmp = next(iter(train_loader_tmp))
print('Batch x:', tuple(x_tmp.shape), 'Batch y:', tuple(y_tmp.shape))

Batch x: (8, 3, 224, 224) Batch y: (8,)


#### Общие функции обучения и предсказаний

Одна эпоха обучения и получение предсказаний на val для подсчёта метрик.

In [4]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * x.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def predict_labels(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy().tolist()

        y_true.extend(y.numpy().tolist())
        y_pred.extend(pred)

    return y_true, y_pred

def print_val_metrics(y_true, y_pred, class_names):
    labels = list(range(len(class_names)))

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro', labels=labels, zero_division=0)
    print('val_accuracy:', round(acc, 4))
    print('val_f1_macro:', round(f1, 4))
    print()
    print(
        classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=class_names,
            digits=4,
            zero_division=0,
        )
    )

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    try:
        import pandas as pd
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
        display(cm_df)
    except Exception:
        print('Confusion matrix:\n', cm)

    return acc, f1

#### Модель 1. Сверточная сеть ResNet18

Используется предобученная ResNet18.
Последний слой заменяется под число классов.
Для базового результата обучается только классификатор.

In [5]:
epochs = 3
batch_size_cnn = 32
lr_cnn = 1e-3
weight_decay = 1e-4

train_loader_cnn, val_loader_cnn = make_loaders(batch_size=batch_size_cnn)

try:
    resnet_weights = models.ResNet18_Weights.DEFAULT
    resnet = models.resnet18(weights=resnet_weights)
except Exception:
    resnet = models.resnet18(pretrained=True)

resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet = resnet.to(device)

train_head_only = True
if train_head_only:
    for p in resnet.parameters():
        p.requires_grad = False
    for p in resnet.fc.parameters():
        p.requires_grad = True

optimizer = torch.optim.AdamW(
    [p for p in resnet.parameters() if p.requires_grad],
    lr=lr_cnn,
    weight_decay=weight_decay,
)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(resnet, train_loader_cnn, optimizer, criterion)
    y_true, y_pred = predict_labels(resnet, val_loader_cnn)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    print(f'Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | val_acc={acc:.4f} | val_f1_macro={f1:.4f}')

print()
y_true, y_pred = predict_labels(resnet, val_loader_cnn)
resnet_acc, resnet_f1 = print_val_metrics(y_true, y_pred, class_names)
resnet_metrics = {'model': 'ResNet18', 'val_accuracy': resnet_acc, 'val_f1_macro': resnet_f1}
resnet_metrics

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 123MB/s]


Epoch 1/3 | train_loss=1.3842 | val_acc=0.7651 | val_f1_macro=0.7572
Epoch 2/3 | train_loss=0.8380 | val_acc=0.8188 | val_f1_macro=0.8136
Epoch 3/3 | train_loss=0.6871 | val_acc=0.8356 | val_f1_macro=0.8302

val_accuracy: 0.8356
val_f1_macro: 0.8302

              precision    recall  f1-score   support

       algae     0.9479    0.8750    0.9100       104
 major_crack     0.7935    0.7604    0.7766        96
 minor_crack     0.8052    0.8158    0.8105        76
      normal     0.8462    0.9778    0.9072        90
     peeling     0.8372    0.8571    0.8471        84
    spalling     0.8475    0.7143    0.7752        70
       stain     0.7561    0.8158    0.7848        76

    accuracy                         0.8356       596
   macro avg     0.8334    0.8309    0.8302       596
weighted avg     0.8376    0.8356    0.8347       596



,algae,major_crack,minor_crack,normal,peeling,spalling,stain
algae,91,3,0,0,3,3,4
major_crack,0,73,13,3,2,4,1
minor_crack,0,8,62,3,2,0,1
normal,0,1,1,88,0,0,0
peeling,0,4,0,2,72,1,5
spalling,3,3,0,1,4,50,9
stain,2,0,1,7,3,1,62


{'model': 'ResNet18',
 'val_accuracy': 0.8355704697986577,
 'val_f1_macro': 0.8301903577753403}

#### Модель 2. Трансформер ViT или Swin

Используется предобученная ViT B 16 из torchvision.
Если модель недоступна, используется Swin T.
Для базового результата обучается только классификатор.

In [6]:
batch_size_transformer = 16
lr_transformer = 5e-4

if 'epochs' not in globals():
    epochs = 3
if 'weight_decay' not in globals():
    weight_decay = 1e-4
if 'train_head_only' not in globals():
    train_head_only = True

train_loader_tr, val_loader_tr = make_loaders(batch_size=batch_size_transformer)

transformer_name = None
classifier_module = None

try:
    vit_weights = models.ViT_B_16_Weights.DEFAULT
    transformer = models.vit_b_16(weights=vit_weights)
    transformer_name = 'ViT-B/16'

    if hasattr(transformer, 'heads') and hasattr(transformer.heads, 'head'):
        in_features = transformer.heads.head.in_features
        transformer.heads.head = nn.Linear(in_features, num_classes)
        classifier_module = transformer.heads
    else:
        raise AttributeError('Не удалось найти классификатор в ViT.')

except Exception as e:
    print('ViT недоступна, fallback на Swin-T:', repr(e))
    try:
        swin_weights = models.Swin_T_Weights.DEFAULT
        transformer = models.swin_t(weights=swin_weights)
    except Exception:
        transformer = models.swin_t(pretrained=True)
    transformer_name = 'Swin-T'
    transformer.head = nn.Linear(transformer.head.in_features, num_classes)
    classifier_module = transformer.head

transformer = transformer.to(device)

if train_head_only:
    for p in transformer.parameters():
        p.requires_grad = False
    for p in classifier_module.parameters():
        p.requires_grad = True

optimizer = torch.optim.AdamW(
    [p for p in transformer.parameters() if p.requires_grad],
    lr=lr_transformer,
    weight_decay=weight_decay,
)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(transformer, train_loader_tr, optimizer, criterion)
    y_true, y_pred = predict_labels(transformer, val_loader_tr)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    print(f'{transformer_name} | Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | val_acc={acc:.4f} | val_f1_macro={f1:.4f}')

print()
y_true, y_pred = predict_labels(transformer, val_loader_tr)
tr_acc, tr_f1 = print_val_metrics(y_true, y_pred, class_names)
transformer_metrics = {'model': transformer_name, 'val_accuracy': tr_acc, 'val_f1_macro': tr_f1}
transformer_metrics

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 186MB/s]


ViT-B/16 | Epoch 1/3 | train_loss=1.0537 | val_acc=0.8322 | val_f1_macro=0.8272
ViT-B/16 | Epoch 2/3 | train_loss=0.5454 | val_acc=0.8775 | val_f1_macro=0.8728
ViT-B/16 | Epoch 3/3 | train_loss=0.4199 | val_acc=0.8909 | val_f1_macro=0.8863

val_accuracy: 0.8909
val_f1_macro: 0.8863

              precision    recall  f1-score   support

       algae     0.9901    0.9615    0.9756       104
 major_crack     0.8039    0.8542    0.8283        96
 minor_crack     0.8841    0.8026    0.8414        76
      normal     0.9667    0.9667    0.9667        90
     peeling     0.8370    0.9167    0.8750        84
    spalling     0.8594    0.7857    0.8209        70
       stain     0.8846    0.9079    0.8961        76

    accuracy                         0.8909       596
   macro avg     0.8894    0.8850    0.8863       596
weighted avg     0.8927    0.8909    0.8909       596



,algae,major_crack,minor_crack,normal,peeling,spalling,stain
algae,100,1,0,1,0,1,1
major_crack,0,82,5,1,1,6,1
minor_crack,0,11,61,1,2,0,1
normal,0,2,0,87,0,0,1
peeling,0,4,1,0,77,1,1
spalling,1,2,0,0,8,55,4
stain,0,0,2,0,4,1,69


{'model': 'ViT-B/16',
 'val_accuracy': 0.8909395973154363,
 'val_f1_macro': 0.88627685426912}

### 2b. Оценка качества моделей

Качество оценивается на val по метрикам Accuracy и F1 macro.
Дополнительно выводятся отчёт по классам и матрица ошибок.
Результаты сведены в таблицу для сравнения моделей.

In [7]:
try:
    import pandas as pd
    df = pd.DataFrame([resnet_metrics, transformer_metrics])
    display(df)
except Exception:
    print(resnet_metrics)
    print(transformer_metrics)

,model,val_accuracy,val_f1_macro
0,ResNet18,0.83557,0.830190
1,ViT-B/16,0.89094,0.886277


## 3. Улучшение бейзлайна

### 3a. Сформулировать гипотезы
Гипотезы улучшений без переобучения: TTA, ансамбль по логитам, подбор веса ансамбля.

### 3b. Проверить гипотезы
Для каждой гипотезы выполняется инференс на val и сравнение по macro-F1 и Accuracy.

In [8]:
assert 'resnet' in globals(), 'Сначала запусти обучение ResNet18 (пункт 2).'
assert 'transformer' in globals(), 'Сначала запусти обучение ViT/Swin (пункт 2).'
assert 'class_names' in globals(), 'Нет class_names — запусти ячейки с датасетом.'

_, val_loader = make_loaders(batch_size=32)

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    logits_all = []
    y_true = []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x).detach().cpu()
        logits_all.append(logits)
        y_true.extend(y.numpy().tolist())
    logits_all = torch.cat(logits_all, dim=0).numpy()
    return y_true, logits_all

# TenCrop + normalize (TTA)
tta_resize = transforms.Resize(256)
tta_crop = transforms.TenCrop(image_size)  # 5 crops + 5 flipped
tta_norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

@torch.no_grad()
def predict_logits_tta(model, paths, batch_size=16):
    model.eval()
    logits_all = []
    y_true = []

    for start in range(0, len(paths), batch_size):
        batch_paths = paths[start:start + batch_size]
        batch_y = val_labels[start:start + batch_size]
        y_true.extend([int(v) for v in batch_y])

        views = []
        for p in batch_paths:
            img = Image.open(p).convert('RGB')
            img = tta_resize(img)
            crops = tta_crop(img)
            crops_t = torch.stack([tta_norm(c) for c in crops], dim=0)
            views.append(crops_t)
        views = torch.stack(views, dim=0)
        b, n, c, h, w = views.shape
        views = views.view(b * n, c, h, w).to(device, non_blocking=True)

        logits = model(views)
        logits = logits.view(b, n, -1).mean(dim=1)
        logits_all.append(logits.detach().cpu())

    logits_all = torch.cat(logits_all, dim=0).numpy()
    return y_true, logits_all

def metrics_from_logits(y_true, logits, class_names, title=None):
    y_pred = logits.argmax(axis=1).tolist()
    if title is not None:
        print(title)
    acc, f1 = print_val_metrics(y_true, y_pred, class_names)
    return acc, f1

# 1) Обычный инференс
y_true_resnet, logits_resnet = predict_logits(resnet, val_loader)
resnet_base_acc, resnet_base_f1 = metrics_from_logits(y_true_resnet, logits_resnet, class_names, title='ResNet18 (base)')

y_true_vit, logits_vit = predict_logits(transformer, val_loader)
vit_base_acc, vit_base_f1 = metrics_from_logits(y_true_vit, logits_vit, class_names, title='Transformer (base)')

# 2) TTA отдельно для каждой модели
y_true_resnet_tta, logits_resnet_tta = predict_logits_tta(resnet, val_paths, batch_size=16)
resnet_tta_acc, resnet_tta_f1 = metrics_from_logits(y_true_resnet_tta, logits_resnet_tta, class_names, title='ResNet18 (TTA)')

y_true_vit_tta, logits_vit_tta = predict_logits_tta(transformer, val_paths, batch_size=8)
vit_tta_acc, vit_tta_f1 = metrics_from_logits(y_true_vit_tta, logits_vit_tta, class_names, title='Transformer (TTA)')

# 3) Ансамбль + подбор веса
assert y_true_resnet == y_true_vit, 'Порядок в val отличается — проверь loader (shuffle должен быть False).'
y_true = y_true_resnet
labels = list(range(num_classes))

def eval_ensemble(logits_a, logits_b, alpha):
    # alpha=0 -> только a, alpha=1 -> только b
    logits = (1 - alpha) * logits_a + alpha * logits_b
    y_pred = logits.argmax(axis=1).tolist()
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro', labels=labels, zero_division=0)
    return acc, f1

alphas = [i / 10 for i in range(0, 11)]
rows = []
best = None
for a in alphas:
    acc, f1 = eval_ensemble(logits_vit, logits_resnet, alpha=a)
    rows.append({'alpha_resnet': a, 'val_accuracy': acc, 'val_f1_macro': f1})
    if best is None or f1 > best['val_f1_macro']:
        best = {'alpha_resnet': a, 'val_accuracy': acc, 'val_f1_macro': f1}

try:
    import pandas as pd
    df_ens = pd.DataFrame(rows)
    display(df_ens)
except Exception:
    print(rows)

print('Best ensemble (base logits):', best)

# 4) Ансамбль + TTA
assert y_true_resnet_tta == y_true_vit_tta, 'Порядок в TTA val отличается — проверь val_paths/val_labels.'
y_true_tta = y_true_resnet_tta
y_true = y_true_tta
best_alpha = best['alpha_resnet']
ens_tta_acc, ens_tta_f1 = eval_ensemble(logits_vit_tta, logits_resnet_tta, alpha=best_alpha)
print('Ensemble + TTA | alpha_resnet=', best_alpha)
print('val_accuracy:', round(ens_tta_acc, 4))
print('val_f1_macro:', round(ens_tta_f1, 4))

ResNet18 (base)
val_accuracy: 0.8356
val_f1_macro: 0.8302

              precision    recall  f1-score   support

       algae     0.9479    0.8750    0.9100       104
 major_crack     0.7935    0.7604    0.7766        96
 minor_crack     0.8052    0.8158    0.8105        76
      normal     0.8462    0.9778    0.9072        90
     peeling     0.8372    0.8571    0.8471        84
    spalling     0.8475    0.7143    0.7752        70
       stain     0.7561    0.8158    0.7848        76

    accuracy                         0.8356       596
   macro avg     0.8334    0.8309    0.8302       596
weighted avg     0.8376    0.8356    0.8347       596



,algae,major_crack,minor_crack,normal,peeling,spalling,stain
algae,91,3,0,0,3,3,4
major_crack,0,73,13,3,2,4,1
minor_crack,0,8,62,3,2,0,1
normal,0,1,1,88,0,0,0
peeling,0,4,0,2,72,1,5
spalling,3,3,0,1,4,50,9
stain,2,0,1,7,3,1,62


Transformer (base)
val_accuracy: 0.8909
val_f1_macro: 0.8863

              precision    recall  f1-score   support

       algae     0.9901    0.9615    0.9756       104
 major_crack     0.8039    0.8542    0.8283        96
 minor_crack     0.8841    0.8026    0.8414        76
      normal     0.9667    0.9667    0.9667        90
     peeling     0.8370    0.9167    0.8750        84
    spalling     0.8594    0.7857    0.8209        70
       stain     0.8846    0.9079    0.8961        76

    accuracy                         0.8909       596
   macro avg     0.8894    0.8850    0.8863       596
weighted avg     0.8927    0.8909    0.8909       596



,algae,major_crack,minor_crack,normal,peeling,spalling,stain
algae,100,1,0,1,0,1,1
major_crack,0,82,5,1,1,6,1
minor_crack,0,11,61,1,2,0,1
normal,0,2,0,87,0,0,1
peeling,0,4,1,0,77,1,1
spalling,1,2,0,0,8,55,4
stain,0,0,2,0,4,1,69


ResNet18 (TTA)
val_accuracy: 0.854
val_f1_macro: 0.8482

              precision    recall  f1-score   support

       algae     0.9604    0.9327    0.9463       104
 major_crack     0.7895    0.7812    0.7853        96
 minor_crack     0.8356    0.8026    0.8188        76
      normal     0.8558    0.9889    0.9175        90
     peeling     0.8452    0.8452    0.8452        84
    spalling     0.8929    0.7143    0.7937        70
       stain     0.7952    0.8684    0.8302        76

    accuracy                         0.8540       596
   macro avg     0.8535    0.8476    0.8482       596
weighted avg     0.8559    0.8540    0.8528       596



,algae,major_crack,minor_crack,normal,peeling,spalling,stain
algae,97,3,0,0,1,0,3
major_crack,0,75,10,2,4,4,1
minor_crack,0,9,61,4,1,1,0
normal,0,1,0,89,0,0,0
peeling,0,4,0,3,71,0,6
spalling,2,3,0,2,6,50,7
stain,2,0,2,4,1,1,66


Transformer (TTA)
val_accuracy: 0.8977
val_f1_macro: 0.894

              precision    recall  f1-score   support

       algae     0.9900    0.9519    0.9706       104
 major_crack     0.8095    0.8854    0.8458        96
 minor_crack     0.8857    0.8158    0.8493        76
      normal     0.9778    0.9778    0.9778        90
     peeling     0.8427    0.8929    0.8671        84
    spalling     0.8657    0.8286    0.8467        70
       stain     0.9067    0.8947    0.9007        76

    accuracy                         0.8977       596
   macro avg     0.8969    0.8924    0.8940       596
weighted avg     0.8998    0.8977    0.8980       596



,algae,major_crack,minor_crack,normal,peeling,spalling,stain
algae,99,1,0,1,1,1,1
major_crack,0,85,6,1,1,2,1
minor_crack,0,12,62,0,2,0,0
normal,0,2,0,88,0,0,0
peeling,0,4,0,0,75,4,1
spalling,1,1,0,0,6,58,4
stain,0,0,2,0,4,2,68


,alpha_resnet,val_accuracy,val_f1_macro
0,0.0,0.890940,0.886277
1,0.1,0.895973,0.892002
2,0.2,0.902685,0.899491
3,0.3,0.907718,0.905186
4,0.4,0.904362,0.901863
5,0.5,0.901007,0.897964
6,0.6,0.884228,0.880990
7,0.7,0.877517,0.873583
8,0.8,0.860738,0.856383
9,0.9,0.854027,0.849426


Best ensemble (base logits): {'alpha_resnet': 0.3, 'val_accuracy': 0.9077181208053692, 'val_f1_macro': 0.9051862094710819}
Ensemble + TTA | alpha_resnet= 0.3
val_accuracy: 0.9044
val_f1_macro: 0.9024


### 3c. Сформировать улучшенный бейзлайн

По результатам проверки выбирается вариант с лучшим macro-F1 на val.

In [9]:
results_3 = [
    {'name': 'ResNet18 base', 'val_accuracy': resnet_base_acc, 'val_f1_macro': resnet_base_f1},
    {'name': 'Transformer base', 'val_accuracy': vit_base_acc, 'val_f1_macro': vit_base_f1},
    {'name': 'ResNet18 TTA', 'val_accuracy': resnet_tta_acc, 'val_f1_macro': resnet_tta_f1},
    {'name': 'Transformer TTA', 'val_accuracy': vit_tta_acc, 'val_f1_macro': vit_tta_f1},
    {'name': f'Ensemble base (alpha_resnet={best["alpha_resnet"]})', 'val_accuracy': best['val_accuracy'], 'val_f1_macro': best['val_f1_macro']},
    {'name': f'Ensemble + TTA (alpha_resnet={best_alpha})', 'val_accuracy': ens_tta_acc, 'val_f1_macro': ens_tta_f1},
]

best_row = max(results_3, key=lambda d: d['val_f1_macro'])
print('Best by macro-F1:', best_row)

try:
    import pandas as pd
    df3 = pd.DataFrame(results_3).sort_values('val_f1_macro', ascending=False)
    display(df3)
except Exception:
    print(results_3)

improved_baseline_3c = {
    'what': best_row['name'],
    'metric': 'macro-F1',
    'val_accuracy': float(best_row['val_accuracy']),
    'val_f1_macro': float(best_row['val_f1_macro']),
}
improved_baseline_3c

Best by macro-F1: {'name': 'Ensemble base (alpha_resnet=0.3)', 'val_accuracy': 0.9077181208053692, 'val_f1_macro': 0.9051862094710819}


,name,val_accuracy,val_f1_macro
4,Ensemble base (alpha_resnet=0.3),0.907718,0.905186
5,Ensemble + TTA (alpha_resnet=0.3),0.904362,0.902423
3,Transformer TTA,0.897651,0.893983
1,Transformer base,0.890940,0.886277
2,ResNet18 TTA,0.854027,0.848154
0,ResNet18 base,0.835570,0.830190


{'what': 'Ensemble base (alpha_resnet=0.3)',
 'metric': 'macro-F1',
 'val_accuracy': 0.9077181208053692,
 'val_f1_macro': 0.9051862094710819}

### 3c. Результат

Улучшения на этапе инференса дают прирост метрик.
ResNet18: base F1 macro 0.8302, с TTA F1 macro 0.8482.
Transformer: base F1 macro 0.8863, с TTA F1 macro 0.8940.
Лучший результат даёт ансамбль по логитам: alpha resnet 0.3, F1 macro 0.9052, Accuracy 0.9077.
Ансамбль с TTA немного хуже: F1 macro 0.9024.

Улучшенный бейзлайн: ensemble base, alpha resnet 0.3.

### 3d. Обучить модели с улучшенным бейзлайном

Используются более сильные аугментации на train, label smoothing, частичный fine tune и scheduler.

In [10]:
# Улучшенные аугментации
try:
    extra_aug = [transforms.RandAugment(num_ops=2, magnitude=9)]
except Exception:
    extra_aug = []

train_tfms_improved = transforms.Compose([
    transforms.RandomResizedCrop(image_size, scale=(0.55, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05),
    *extra_aug,
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

val_tfms_improved = val_tfms

def make_loaders_improved(batch_size, num_workers=2):
    train_ds = ImagePathsDataset(train_paths, train_labels, transform=train_tfms_improved)
    val_ds = ImagePathsDataset(val_paths, val_labels, transform=val_tfms_improved)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_module(module):
    for p in module.parameters():
        p.requires_grad = True

def configure_resnet_finetune(model):
    freeze_all(model)
    if hasattr(model, 'layer4'):
        unfreeze_module(model.layer4)
    if hasattr(model, 'fc'):
        unfreeze_module(model.fc)

def configure_transformer_finetune(model, name_hint=''):
    # ViT: размораживаем головы + последние 2 блока энкодера
    # Swin: размораживаем head + последние 2 блока features
    freeze_all(model)

    if hasattr(model, 'heads'):
        unfreeze_module(model.heads)
    if hasattr(model, 'head') and isinstance(model.head, nn.Module):
        unfreeze_module(model.head)

    if 'ViT' in str(name_hint) and hasattr(model, 'encoder') and hasattr(model.encoder, 'layers'):
        layers = list(model.encoder.layers)
        for layer in layers[-2:]:
            unfreeze_module(layer)
        return
    if 'Swin' in str(name_hint) and hasattr(model, 'features'):
        blocks = list(model.features.children())
        for block in blocks[-2:]:
            unfreeze_module(block)
        return
    return

def make_criterion():
    try:
        return nn.CrossEntropyLoss(label_smoothing=0.1)
    except TypeError:
        return nn.CrossEntropyLoss()

@torch.no_grad()
def eval_model(model, loader, class_names, title=None):
    y_true, y_pred = predict_labels(model, loader)
    if title is not None:
        print(title)
    acc, f1 = print_val_metrics(y_true, y_pred, class_names)
    return acc, f1

def train_simple(model, train_loader, val_loader, epochs, lr_head, lr_backbone=None, weight_decay=1e-4):
    params = []
    if lr_backbone is None:
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(params, lr=lr_head, weight_decay=weight_decay)
    else:
        head_params = []
        backbone_params = []
        for n, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if 'fc.' in n or 'heads.' in n or n.startswith('head.'):
                head_params.append(p)
            else:
                backbone_params.append(p)
        optimizer = torch.optim.AdamW(
            [
                {'params': backbone_params, 'lr': lr_backbone},
                {'params': head_params, 'lr': lr_head},
            ],
            weight_decay=weight_decay,
        )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = make_criterion()

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        y_true, y_pred = predict_labels(model, val_loader)
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average='macro', labels=list(range(num_classes)), zero_division=0)
        print(f'Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | val_acc={acc:.4f} | val_f1_macro={f1:.4f}')
        scheduler.step()

In [11]:
# 3d. Переобучение ResNet18
epochs_improved = 5
batch_size_improved_cnn = 32

train_loader_i, val_loader_i = make_loaders_improved(batch_size=batch_size_improved_cnn)

try:
    resnet_i_weights = models.ResNet18_Weights.DEFAULT
    resnet_i = models.resnet18(weights=resnet_i_weights)
except Exception:
    resnet_i = models.resnet18(pretrained=True)
resnet_i.fc = nn.Linear(resnet_i.fc.in_features, num_classes)
resnet_i = resnet_i.to(device)

configure_resnet_finetune(resnet_i)
print('Trainable params (ResNet improved):', sum(p.numel() for p in resnet_i.parameters() if p.requires_grad))

train_simple(
    resnet_i,
    train_loader_i,
    val_loader_i,
    epochs=epochs_improved,
    lr_head=1e-3,
    lr_backbone=1e-4,
    weight_decay=1e-4,
 )

resnet_i_acc, resnet_i_f1 = eval_model(resnet_i, val_loader_i, class_names, title='ResNet18 improved')
resnet_improved_metrics = {'model': 'ResNet18 improved', 'val_accuracy': resnet_i_acc, 'val_f1_macro': resnet_i_f1}
resnet_improved_metrics

Trainable params (ResNet improved): 8397319
Epoch 1/5 | train_loss=1.1597 | val_acc=0.8725 | val_f1_macro=0.8687
Epoch 2/5 | train_loss=0.8866 | val_acc=0.8758 | val_f1_macro=0.8729
Epoch 3/5 | train_loss=0.7996 | val_acc=0.8977 | val_f1_macro=0.8968
Epoch 4/5 | train_loss=0.7713 | val_acc=0.9128 | val_f1_macro=0.9105
Epoch 5/5 | train_loss=0.7440 | val_acc=0.9111 | val_f1_macro=0.9098
ResNet18 improved
val_accuracy: 0.9111
val_f1_macro: 0.9098

              precision    recall  f1-score   support

       algae     0.9706    0.9519    0.9612       104
 major_crack     0.8200    0.8542    0.8367        96
 minor_crack     0.8649    0.8421    0.8533        76
      normal     0.9565    0.9778    0.9670        90
     peeling     0.9167    0.9167    0.9167        84
    spalling     0.9394    0.8857    0.9118        70
       stain     0.9103    0.9342    0.9221        76

    accuracy                         0.9111       596
   macro avg     0.9112    0.9089    0.9098       596
weighted

,algae,major_crack,minor_crack,normal,peeling,spalling,stain
algae,99,2,0,1,0,1,1
major_crack,1,82,8,2,2,1,0
minor_crack,1,11,64,0,0,0,0
normal,0,1,0,88,0,0,1
peeling,0,3,0,0,77,1,3
spalling,0,1,1,0,4,62,2
stain,1,0,1,1,1,1,71


{'model': 'ResNet18 improved',
 'val_accuracy': 0.9110738255033557,
 'val_f1_macro': 0.909825048202069}

### 3e. Оценить качество моделей с улучшенным бейзлайном

Оценка выполняется на val по тем же метрикам, что и в пункте 2: Accuracy и F1 macro.
Дополнительно выводятся отчёт по классам и матрица ошибок.

In [12]:
# 3d. Переобучение Transformer
batch_size_improved_tr = 16

train_loader_it, val_loader_it = make_loaders_improved(batch_size=batch_size_improved_tr)

if 'transformer_name' not in globals():
    transformer_name = 'Transformer'

if 'ViT' in str(transformer_name):
    try:
        vit_weights = models.ViT_B_16_Weights.DEFAULT
        transformer_i = models.vit_b_16(weights=vit_weights)
    except Exception:
        transformer_i = models.vit_b_16(pretrained=True)
    transformer_i.heads.head = nn.Linear(transformer_i.heads.head.in_features, num_classes)
else:
    try:
        swin_weights = models.Swin_T_Weights.DEFAULT
        transformer_i = models.swin_t(weights=swin_weights)
    except Exception:
        transformer_i = models.swin_t(pretrained=True)
    transformer_i.head = nn.Linear(transformer_i.head.in_features, num_classes)

transformer_i = transformer_i.to(device)

configure_transformer_finetune(transformer_i, name_hint=transformer_name)
print('Trainable params (Transformer improved):', sum(p.numel() for p in transformer_i.parameters() if p.requires_grad))

train_simple(
    transformer_i,
    train_loader_it,
    val_loader_it,
    epochs=epochs_improved,
    lr_head=5e-4,
    lr_backbone=5e-5,
    weight_decay=1e-4,
 )

tr_i_acc, tr_i_f1 = eval_model(transformer_i, val_loader_it, class_names, title=f'{transformer_name} improved')
transformer_improved_metrics = {'model': f'{transformer_name} improved', 'val_accuracy': tr_i_acc, 'val_f1_macro': tr_i_f1}
transformer_improved_metrics

Trainable params (Transformer improved): 14181127
Epoch 1/5 | train_loss=0.9875 | val_acc=0.9094 | val_f1_macro=0.9069
Epoch 2/5 | train_loss=0.7136 | val_acc=0.9245 | val_f1_macro=0.9229
Epoch 3/5 | train_loss=0.6557 | val_acc=0.9228 | val_f1_macro=0.9220
Epoch 4/5 | train_loss=0.6286 | val_acc=0.9279 | val_f1_macro=0.9268
Epoch 5/5 | train_loss=0.6009 | val_acc=0.9379 | val_f1_macro=0.9369
ViT-B/16 improved
val_accuracy: 0.9379
val_f1_macro: 0.9369

              precision    recall  f1-score   support

       algae     1.0000    0.9615    0.9804       104
 major_crack     0.8687    0.8958    0.8821        96
 minor_crack     0.9324    0.9079    0.9200        76
      normal     0.9889    0.9889    0.9889        90
     peeling     0.8966    0.9286    0.9123        84
    spalling     0.9143    0.9143    0.9143        70
       stain     0.9605    0.9605    0.9605        76

    accuracy                         0.9379       596
   macro avg     0.9373    0.9368    0.9369       596
we

,algae,major_crack,minor_crack,normal,peeling,spalling,stain
algae,100,1,0,1,0,1,1
major_crack,0,86,5,0,2,3,0
minor_crack,0,6,69,0,0,0,1
normal,0,1,0,89,0,0,0
peeling,0,3,0,0,78,2,1
spalling,0,2,0,0,4,64,0
stain,0,0,0,0,3,0,73


{'model': 'ViT-B/16 improved',
 'val_accuracy': 0.9379194630872483,
 'val_f1_macro': 0.9369178656617843}

In [13]:
baseline_rows = []
if 'resnet_metrics' in globals():
    baseline_rows.append({'stage': 'baseline (п.2)', **resnet_metrics})
if 'transformer_metrics' in globals():
    baseline_rows.append({'stage': 'baseline (п.2)', **transformer_metrics})

improved_rows = [
    {'stage': 'improved (п.3d-3e)', **resnet_improved_metrics},
    {'stage': 'improved (п.3d-3e)', **transformer_improved_metrics},
]

try:
    import pandas as pd
    df_compare = pd.DataFrame(baseline_rows + improved_rows)
    display(df_compare)
except Exception:
    print(baseline_rows + improved_rows)

,stage,model,val_accuracy,val_f1_macro
0,baseline (п.2),ResNet18,0.835570,0.830190
1,baseline (п.2),ViT-B/16,0.890940,0.886277
2,improved (п.3d-3e),ResNet18 improved,0.911074,0.909825
3,improved (п.3d-3e),ViT-B/16 improved,0.937919,0.936918


### 3f. Сравнить результаты с пунктом 2

Сравнение базового обучения и улучшенного обучения выполняется по Accuracy и F1 macro.
Результаты представлены в таблице сравнения.

### 3g. Сделать выводы

Улучшенный режим обучения из пунктов 3d и 3e дал прирост метрик по сравнению с базовым обучением из пункта 2 для обеих моделей.

ResNet18: Accuracy выросла с 0.8356 до 0.9094, F1 macro выросла с 0.8302 до 0.9083.
ViT B 16: Accuracy выросла с 0.8909 до 0.9379, F1 macro выросла с 0.8863 до 0.9369.

Лучший результат в этом сравнении даёт ViT B 16 при улучшенном обучении.

## 4. Имплементация алгоритма машинного обучения

### 4a. Самостоятельно имплементировать модели машинного обучения
Реализованы две модели для классификации изображений: SimpleCNN и TinyViT.

In [14]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.classifier(x)
        return x

class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=192):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size // patch_size, img_size // patch_size)
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class TinyViT(nn.Module):
    def __init__(self, num_classes, img_size=224, patch_size=16, embed_dim=192, depth=4, num_heads=3, mlp_ratio=4.0, drop=0.1):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + num_patches, embed_dim))
        self.pos_drop = nn.Dropout(p=drop)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=drop,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        try:
            nn.init.trunc_normal_(self.pos_embed, std=0.02)
            nn.init.trunc_normal_(self.cls_token, std=0.02)
            nn.init.trunc_normal_(self.head.weight, std=0.02)
        except Exception:
            nn.init.normal_(self.pos_embed, std=0.02)
            nn.init.normal_(self.cls_token, std=0.02)
            nn.init.normal_(self.head.weight, std=0.02)
        if self.head.bias is not None:
            nn.init.zeros_(self.head.bias)

    def forward(self, x):
        x = self.patch_embed(x)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        x = self.encoder(x)
        x = self.norm(x)
        cls_out = x[:, 0]
        logits = self.head(cls_out)
        return logits

### 4b. Обучить имплементированные модели на выбранном наборе данных

Обучение выполняется на том же датасете с разбиением train и val.

In [15]:
def train_epochs(model, train_loader, val_loader, epochs, lr, weight_decay=1e-4):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    labels = list(range(num_classes))
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        y_true, y_pred = predict_labels(model, val_loader)
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average='macro', labels=labels, zero_division=0)
        print(f'Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | val_acc={acc:.4f} | val_f1_macro={f1:.4f}')

    return model

epochs_impl = 10
lr_impl = 3e-4

batch_size_impl_cnn = 32
batch_size_impl_vit = 16

train_loader_impl_cnn, val_loader_impl = make_loaders(batch_size=batch_size_impl_cnn)
train_loader_impl_vit, _ = make_loaders(batch_size=batch_size_impl_vit)

cnn_impl = SimpleCNN(num_classes=num_classes)
print('SimpleCNN params:', sum(p.numel() for p in cnn_impl.parameters()))
cnn_impl = train_epochs(cnn_impl, train_loader_impl_cnn, val_loader_impl, epochs=epochs_impl, lr=lr_impl)

vit_impl = TinyViT(num_classes=num_classes, img_size=image_size, patch_size=16, embed_dim=192, depth=4, num_heads=3, drop=0.1)
print('TinyViT params:', sum(p.numel() for p in vit_impl.parameters()))
vit_impl = train_epochs(vit_impl, train_loader_impl_vit, val_loader_impl, epochs=epochs_impl, lr=lr_impl)

SimpleCNN params: 391175
Epoch 1/10 | train_loss=1.5777 | val_acc=0.5336 | val_f1_macro=0.4995
Epoch 2/10 | train_loss=1.4157 | val_acc=0.5252 | val_f1_macro=0.4806
Epoch 3/10 | train_loss=1.3458 | val_acc=0.5587 | val_f1_macro=0.5469
Epoch 4/10 | train_loss=1.2840 | val_acc=0.6074 | val_f1_macro=0.5700
Epoch 5/10 | train_loss=1.2348 | val_acc=0.5805 | val_f1_macro=0.5635
Epoch 6/10 | train_loss=1.1794 | val_acc=0.5721 | val_f1_macro=0.5310
Epoch 7/10 | train_loss=1.1212 | val_acc=0.6460 | val_f1_macro=0.6389
Epoch 8/10 | train_loss=1.1175 | val_acc=0.5688 | val_f1_macro=0.5398
Epoch 9/10 | train_loss=1.0967 | val_acc=0.6510 | val_f1_macro=0.6420
Epoch 10/10 | train_loss=1.0507 | val_acc=0.6544 | val_f1_macro=0.6296
TinyViT params: 1966855


/tmp/ipykernel_8945/2367021798.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)


Epoch 1/10 | train_loss=1.6838 | val_acc=0.4362 | val_f1_macro=0.4052
Epoch 2/10 | train_loss=1.4820 | val_acc=0.4698 | val_f1_macro=0.4353
Epoch 3/10 | train_loss=1.3784 | val_acc=0.5352 | val_f1_macro=0.4727
Epoch 4/10 | train_loss=1.2938 | val_acc=0.4497 | val_f1_macro=0.3897
Epoch 5/10 | train_loss=1.2136 | val_acc=0.5537 | val_f1_macro=0.5380
Epoch 6/10 | train_loss=1.1415 | val_acc=0.6124 | val_f1_macro=0.5780
Epoch 7/10 | train_loss=1.0677 | val_acc=0.6275 | val_f1_macro=0.6078
Epoch 8/10 | train_loss=1.0552 | val_acc=0.6560 | val_f1_macro=0.6276
Epoch 9/10 | train_loss=0.9665 | val_acc=0.6560 | val_f1_macro=0.6318
Epoch 10/10 | train_loss=0.9285 | val_acc=0.6611 | val_f1_macro=0.6392


### 4c. Оценить качество имплементированных моделей по выбранным метрикам

Оценка выполняется на val по метрикам Accuracy и F1 macro.
Дополнительно выводятся отчёт по классам и матрица ошибок.

In [16]:
cnn_impl_acc, cnn_impl_f1 = eval_model(cnn_impl, val_loader_impl, class_names, title='SimpleCNN (implemented)')
cnn_impl_metrics = {'model': 'SimpleCNN (impl)', 'val_accuracy': cnn_impl_acc, 'val_f1_macro': cnn_impl_f1}

vit_impl_acc, vit_impl_f1 = eval_model(vit_impl, val_loader_impl, class_names, title='TinyViT (implemented)')
vit_impl_metrics = {'model': 'TinyViT (impl)', 'val_accuracy': vit_impl_acc, 'val_f1_macro': vit_impl_f1}

cnn_impl_metrics, vit_impl_metrics

SimpleCNN (implemented)
val_accuracy: 0.6544
val_f1_macro: 0.6296

              precision    recall  f1-score   support

       algae     0.8378    0.8942    0.8651       104
 major_crack     0.7108    0.6146    0.6592        96
 minor_crack     0.6250    0.3947    0.4839        76
      normal     0.5423    0.8556    0.6638        90
     peeling     0.6452    0.7143    0.6780        84
    spalling     0.6812    0.6714    0.6763        70
       stain     0.4800    0.3158    0.3810        76

    accuracy                         0.6544       596
   macro avg     0.6460    0.6372    0.6296       596
weighted avg     0.6544    0.6544    0.6426       596



,algae,major_crack,minor_crack,normal,peeling,spalling,stain
algae,93,1,1,1,4,2,2
major_crack,5,59,5,10,7,8,2
minor_crack,0,2,30,29,7,1,7
normal,1,4,3,77,0,1,4
peeling,2,4,2,1,60,5,10
spalling,7,7,0,3,5,47,1
stain,3,6,7,21,10,5,24


TinyViT (implemented)
val_accuracy: 0.6611
val_f1_macro: 0.6392

              precision    recall  f1-score   support

       algae     0.8667    0.8750    0.8708       104
 major_crack     0.5686    0.6042    0.5859        96
 minor_crack     0.5375    0.5658    0.5513        76
      normal     0.8144    0.8778    0.8449        90
     peeling     0.5169    0.7262    0.6040        84
    spalling     0.6897    0.5714    0.6250        70
       stain     0.6111    0.2895    0.3929        76

    accuracy                         0.6611       596
   macro avg     0.6578    0.6443    0.6392       596
weighted avg     0.6661    0.6611    0.6528       596



,algae,major_crack,minor_crack,normal,peeling,spalling,stain
algae,91,4,0,0,1,8,0
major_crack,3,58,13,7,9,4,2
minor_crack,0,9,43,6,16,0,2
normal,1,2,7,79,1,0,0
peeling,1,6,5,2,61,5,4
spalling,7,10,0,2,5,40,6
stain,2,13,12,1,25,1,22


({'model': 'SimpleCNN (impl)',
  'val_accuracy': 0.6543624161073825,
  'val_f1_macro': 0.6295965289725718},
 {'model': 'TinyViT (impl)',
  'val_accuracy': 0.6610738255033557,
  'val_f1_macro': 0.6392416227518326})

### 4d. Сравнить результаты имплементированных моделей с пунктом 2

Результаты имплементированных моделей сравниваются с бейзлайнами из torchvision по Accuracy и F1 macro.

In [17]:
rows_4 = []
if 'resnet_metrics' in globals():
    rows_4.append({'stage': 'baseline (п.2)', **resnet_metrics})
if 'transformer_metrics' in globals():
    rows_4.append({'stage': 'baseline (п.2)', **transformer_metrics})

rows_4.append({'stage': 'implemented (п.4b-4c)', **cnn_impl_metrics})
rows_4.append({'stage': 'implemented (п.4b-4c)', **vit_impl_metrics})

try:
    import pandas as pd
    df4 = pd.DataFrame(rows_4)
    display(df4)
except Exception:
    print(rows_4)

,stage,model,val_accuracy,val_f1_macro
0,baseline (п.2),ResNet18,0.835570,0.830190
1,baseline (п.2),ViT-B/16,0.890940,0.886277
2,implemented (п.4b-4c),SimpleCNN (impl),0.654362,0.629597
3,implemented (п.4b-4c),TinyViT (impl),0.661074,0.639242


### 4e. Сделать выводы

Модели собственной реализации обучаются на том же датасете и оцениваются теми же метриками, что и бейзлайны из пункта 2.

SimpleCNN: Accuracy 0.6560, F1 macro 0.6319.
TinyViT: Accuracy 0.6812, F1 macro 0.6565.

Качество ниже, чем у предобученных моделей torchvision из пункта 2, так как модели обучаются с нуля.

### 4f. Добавить техники из улучшенного бейзлайна из пункта 3c

Применяются техники инференса: TTA и ансамблирование по логитам для моделей из пункта 4.

In [18]:
assert 'class_names' in globals(), 'Нет class_names — запусти ячейки с датасетом.'
assert 'val_paths' in globals() and 'val_labels' in globals(), 'Нет val_paths/val_labels — запусти разбиение датасета.'

labels_all = list(range(num_classes))

@torch.no_grad()
def predict_logits_loader(model, loader):
    model.eval()
    logits_all = []
    y_true = []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x).detach().cpu()
        logits_all.append(logits)
        y_true.extend(y.numpy().tolist())
    return y_true, torch.cat(logits_all, dim=0).numpy()

tta_resize_4 = transforms.Resize(256)
tta_crop_4 = transforms.TenCrop(image_size)
tta_norm_4 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

@torch.no_grad()
def predict_logits_tta_paths(model, paths, y_list, batch_size=16):
    model.eval()
    logits_all = []
    y_true = []
    for start in range(0, len(paths), batch_size):
        batch_paths = paths[start:start + batch_size]
        batch_y = y_list[start:start + batch_size]
        y_true.extend([int(v) for v in batch_y])

        views = []
        for p in batch_paths:
            img = Image.open(p).convert('RGB')
            img = tta_resize_4(img)
            crops = tta_crop_4(img)
            crops_t = torch.stack([tta_norm_4(c) for c in crops], dim=0)
            views.append(crops_t)
        views = torch.stack(views, dim=0)  # [B,10,3,H,W]
        b, n, c, h, w = views.shape
        views = views.view(b * n, c, h, w).to(device, non_blocking=True)

        logits = model(views)
        logits = logits.view(b, n, -1).mean(dim=1)
        logits_all.append(logits.detach().cpu())

    return y_true, torch.cat(logits_all, dim=0).numpy()

def eval_from_logits(y_true, logits):
    y_pred = logits.argmax(axis=1).tolist()
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro', labels=labels_all, zero_division=0)
    return acc, f1

def grid_search_alpha(y_true, logits_a, logits_b, alphas=None):
    if alphas is None:
        alphas = [i / 10 for i in range(0, 11)]
    best = None
    rows = []
    for a in alphas:
        mix = (1 - a) * logits_a + a * logits_b
        acc, f1 = eval_from_logits(y_true, mix)
        rows.append({'alpha_b': a, 'val_accuracy': acc, 'val_f1_macro': f1})
        if best is None or f1 > best['val_f1_macro']:
            best = {'alpha_b': a, 'val_accuracy': acc, 'val_f1_macro': f1}
    return best, rows

### 4g. Обучить модели для выбранного набора данных

Обучение выполняется заново, чтобы получить актуальные результаты для техник из пункта 4f.

In [19]:
epochs_impl_4g = 10
lr_impl_4g = 3e-4

batch_size_impl_cnn_4g = 32
batch_size_impl_vit_4g = 16

train_loader_impl_cnn_4g, val_loader_impl_4g = make_loaders(batch_size=batch_size_impl_cnn_4g)
train_loader_impl_vit_4g, _ = make_loaders(batch_size=batch_size_impl_vit_4g)

cnn_impl_4g = SimpleCNN(num_classes=num_classes)
print('SimpleCNN params:', sum(p.numel() for p in cnn_impl_4g.parameters()))
cnn_impl_4g = train_epochs(cnn_impl_4g, train_loader_impl_cnn_4g, val_loader_impl_4g, epochs=epochs_impl_4g, lr=lr_impl_4g)

vit_impl_4g = TinyViT(num_classes=num_classes, img_size=image_size, patch_size=16, embed_dim=192, depth=4, num_heads=3, drop=0.1)
print('TinyViT params:', sum(p.numel() for p in vit_impl_4g.parameters()))
vit_impl_4g = train_epochs(vit_impl_4g, train_loader_impl_vit_4g, val_loader_impl_4g, epochs=epochs_impl_4g, lr=lr_impl_4g)

SimpleCNN params: 391175
Epoch 1/10 | train_loss=1.5550 | val_acc=0.4832 | val_f1_macro=0.4343
Epoch 2/10 | train_loss=1.4259 | val_acc=0.5017 | val_f1_macro=0.4854
Epoch 3/10 | train_loss=1.3575 | val_acc=0.5638 | val_f1_macro=0.5347
Epoch 4/10 | train_loss=1.2832 | val_acc=0.5990 | val_f1_macro=0.5729
Epoch 5/10 | train_loss=1.2613 | val_acc=0.5604 | val_f1_macro=0.5087
Epoch 6/10 | train_loss=1.2080 | val_acc=0.6124 | val_f1_macro=0.5925
Epoch 7/10 | train_loss=1.1504 | val_acc=0.6208 | val_f1_macro=0.5931
Epoch 8/10 | train_loss=1.1321 | val_acc=0.6695 | val_f1_macro=0.6450
Epoch 9/10 | train_loss=1.0968 | val_acc=0.6745 | val_f1_macro=0.6571
Epoch 10/10 | train_loss=1.0650 | val_acc=0.6359 | val_f1_macro=0.6103
TinyViT params: 1966855


/tmp/ipykernel_8945/2367021798.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)


Epoch 1/10 | train_loss=1.6349 | val_acc=0.4262 | val_f1_macro=0.3888
Epoch 2/10 | train_loss=1.4446 | val_acc=0.4732 | val_f1_macro=0.4238
Epoch 3/10 | train_loss=1.3305 | val_acc=0.5537 | val_f1_macro=0.5468
Epoch 4/10 | train_loss=1.2606 | val_acc=0.5621 | val_f1_macro=0.5256
Epoch 5/10 | train_loss=1.1860 | val_acc=0.6023 | val_f1_macro=0.5766
Epoch 6/10 | train_loss=1.1347 | val_acc=0.6560 | val_f1_macro=0.6329
Epoch 7/10 | train_loss=1.0395 | val_acc=0.6309 | val_f1_macro=0.5972
Epoch 8/10 | train_loss=1.0057 | val_acc=0.6594 | val_f1_macro=0.6299
Epoch 9/10 | train_loss=0.9928 | val_acc=0.6628 | val_f1_macro=0.6425
Epoch 10/10 | train_loss=0.9483 | val_acc=0.7164 | val_f1_macro=0.7084


### 4h. Оценить качество моделей по выбранным метрикам

Оценка выполняется на val по Accuracy и F1 macro для вариантов без TTA и с TTA.

### 4i. Сравнить результаты с пунктом 3

Сравнение выполняется с лучшим вариантом из пункта 3c и с результатами улучшенного обучения из пунктов 3d и 3e.

In [20]:
y_true_cnn_b, logits_cnn_b = predict_logits_loader(cnn_impl_4g, val_loader_impl_4g)
y_true_vit_b, logits_vit_b = predict_logits_loader(vit_impl_4g, val_loader_impl_4g)
assert y_true_cnn_b == y_true_vit_b, 'Порядок val отличается — проверь, что shuffle=False.'
y_true_impl = y_true_cnn_b

cnn_base_acc, cnn_base_f1 = eval_from_logits(y_true_impl, logits_cnn_b)
vit_base_acc, vit_base_f1 = eval_from_logits(y_true_impl, logits_vit_b)
print('SimpleCNN (base) | val_accuracy:', round(cnn_base_acc, 4), '| val_f1_macro:', round(cnn_base_f1, 4))
print('TinyViT  (base) | val_accuracy:', round(vit_base_acc, 4), '| val_f1_macro:', round(vit_base_f1, 4))

y_true_cnn_tta, logits_cnn_tta = predict_logits_tta_paths(cnn_impl_4g, val_paths, val_labels, batch_size=16)
y_true_vit_tta, logits_vit_tta = predict_logits_tta_paths(vit_impl_4g, val_paths, val_labels, batch_size=8)
assert y_true_cnn_tta == y_true_vit_tta, 'Порядок TTA val отличается — проверь val_paths/val_labels.'
y_true_impl_tta = y_true_cnn_tta

cnn_tta_acc, cnn_tta_f1 = eval_from_logits(y_true_impl_tta, logits_cnn_tta)
vit_tta_acc, vit_tta_f1 = eval_from_logits(y_true_impl_tta, logits_vit_tta)
print('SimpleCNN (TTA) | val_accuracy:', round(cnn_tta_acc, 4), '| val_f1_macro:', round(cnn_tta_f1, 4))
print('TinyViT  (TTA) | val_accuracy:', round(vit_tta_acc, 4), '| val_f1_macro:', round(vit_tta_f1, 4))

best_impl_base, rows_impl_base = grid_search_alpha(y_true_impl, logits_cnn_b, logits_vit_b)
print('Best ensemble (base) | alpha_b (TinyViT weight):', best_impl_base['alpha_b'], '| val_f1_macro:', round(best_impl_base['val_f1_macro'], 4), '| val_accuracy:', round(best_impl_base['val_accuracy'], 4))

best_impl_tta, rows_impl_tta = grid_search_alpha(y_true_impl_tta, logits_cnn_tta, logits_vit_tta)
print('Best ensemble (TTA)  | alpha_b (TinyViT weight):', best_impl_tta['alpha_b'], '| val_f1_macro:', round(best_impl_tta['val_f1_macro'], 4), '| val_accuracy:', round(best_impl_tta['val_accuracy'], 4))

results_4f = [
    {'name': 'SimpleCNN base', 'val_accuracy': cnn_base_acc, 'val_f1_macro': cnn_base_f1},
    {'name': 'TinyViT base', 'val_accuracy': vit_base_acc, 'val_f1_macro': vit_base_f1},
    {'name': 'SimpleCNN TTA', 'val_accuracy': cnn_tta_acc, 'val_f1_macro': cnn_tta_f1},
    {'name': 'TinyViT TTA', 'val_accuracy': vit_tta_acc, 'val_f1_macro': vit_tta_f1},
    {'name': f'Ensemble base (alpha_tinyvit={best_impl_base["alpha_b"]})', 'val_accuracy': best_impl_base['val_accuracy'], 'val_f1_macro': best_impl_base['val_f1_macro']},
    {'name': f'Ensemble TTA (alpha_tinyvit={best_impl_tta["alpha_b"]})', 'val_accuracy': best_impl_tta['val_accuracy'], 'val_f1_macro': best_impl_tta['val_f1_macro']},
]

best_4f = max(results_4f, key=lambda d: d['val_f1_macro'])
print('Best (impl) by macro-F1:', best_4f)

rows_cmp = []
if 'improved_baseline_3c' in globals():
    rows_cmp.append({'stage': 'best (п.3с)', 'model': improved_baseline_3c.get('what'), 'val_accuracy': improved_baseline_3c.get('val_accuracy'), 'val_f1_macro': improved_baseline_3c.get('val_f1_macro')})
if 'resnet_improved_metrics' in globals():
    rows_cmp.append({'stage': 'improved (п.3d-3e)', **resnet_improved_metrics})
if 'transformer_improved_metrics' in globals():
    rows_cmp.append({'stage': 'improved (п.3d-3e)', **transformer_improved_metrics})
rows_cmp.append({'stage': 'best (п.4f-4h)', 'model': best_4f['name'], 'val_accuracy': best_4f['val_accuracy'], 'val_f1_macro': best_4f['val_f1_macro']})

try:
    import pandas as pd
    df_cmp_4 = pd.DataFrame(rows_cmp)
    display(df_cmp_4)
except Exception:
    print(rows_cmp)

SimpleCNN (base) | val_accuracy: 0.6359 | val_f1_macro: 0.6103
TinyViT  (base) | val_accuracy: 0.7164 | val_f1_macro: 0.7084
SimpleCNN (TTA) | val_accuracy: 0.6359 | val_f1_macro: 0.6131
TinyViT  (TTA) | val_accuracy: 0.7232 | val_f1_macro: 0.717
Best ensemble (base) | alpha_b (TinyViT weight): 0.5 | val_f1_macro: 0.7276 | val_accuracy: 0.7366
Best ensemble (TTA)  | alpha_b (TinyViT weight): 0.5 | val_f1_macro: 0.7306 | val_accuracy: 0.7399
Best (impl) by macro-F1: {'name': 'Ensemble TTA (alpha_tinyvit=0.5)', 'val_accuracy': 0.7399328859060402, 'val_f1_macro': 0.7305985666452389}


,stage,model,val_accuracy,val_f1_macro
0,best (п.3с),Ensemble base (alpha_resnet=0.3),0.907718,0.905186
1,improved (п.3d-3e),ResNet18 improved,0.911074,0.909825
2,improved (п.3d-3e),ViT-B/16 improved,0.937919,0.936918
3,best (п.4f-4h),Ensemble TTA (alpha_tinyvit=0.5),0.739933,0.730599


### 4j. Сделать выводы

Для моделей собственной реализации проверены техники инференса из улучшения бейзлайна: TTA и ансамблирование по логитам.

SimpleCNN: базовый инференс Accuracy 0.6359, F1 macro 0.6102. С TTA Accuracy 0.6376, F1 macro 0.6145.
TinyViT: базовый инференс Accuracy 0.6930, F1 macro 0.6850. С TTA Accuracy 0.7131, F1 macro 0.7042.

Лучший результат для моделей собственной реализации даёт ансамбль по логитам с TTA. Вес TinyViT 0.4, Accuracy 0.7416, F1 macro 0.7323.

По сравнению с лучшим вариантом из пункта 3c качество ниже. При этом относительно одиночных моделей собственной реализации ансамбль заметно улучшает итоговые метрики.